# 02 · Detection with torchvision —— 预训练推理、可视化与指标

**家族位置**：`03_CNN_Segmentation_Detection` 第 2 站（检测）。上一站自训了分割（每像素一签，mIoU/Dice）；本章换到**检测**（每目标一框+一类），任务粒度、指标、数据形态再次换挡——IoU→框 IoU、mAP 代替 mIoU，Anchor/NMS 登场。

**学习目标**
1. two-stage（Faster R-CNN：RPN→RoI）vs one-stage（RetinaNet：Focal Loss）技术路线与取舍
2. 检测三件套：Anchor / IoU / NMS——手写 NMS 现场验证
3. 预训练推理全流程：torchvision 权重加载→阈值过滤→可视化→逐图解读
4. 评估体系：mAP 的直观含义（为什么检测不能只看 accuracy）

## 1. 原理：从分割到检测

### 通俗理解

**一句话**：分割给每个像素贴标签，检测给每个物体画框+贴标签。分割错一个像素只是一点，检测错一个框就是漏/误检一个物体。

**比喻**：分割是"涂色本"（逐像素涂），检测是"点名+画圈"（这有只猫、那有条狗，圈出来）。圈得准不准用**框 IoU**量，点得全不全用 **mAP** 量。

### 两条路线

```
Faster R-CNN (two-stage)：  backbone→FPN ─┬─→ RPN(Anchor 粗筛 ~2000 框) ─→ RoI Head(精修+分类) 
                                          └─→ 共享特征，两步走，先提候选再精判

RetinaNet (one-stage)：     backbone→FPN ──→ 每层 Anchor 直接回 归+分类（Focal Loss 压制易分负样本）
                                          └─→ 一步到位，速度快，Focal 解决"背景太多"不均衡

Mask R-CNN：               Faster R-CNN + Mask Head（实例分割 = 检测框 + 像素掩码）
```

### 指标与组件

- **框 IoU**：两框交/并，>0.5 算命中
- **NMS**：同一物体多个框，按分数排序，IoU>阈值（0.5）的低分框删掉
- **mAP**：多类 AP 均值；AP = PR 曲线下面积（召回从 0→1 时精度的平均）
- **Anchor**：预设多尺度多比例的参考框，网络学"相对 Anchor 的偏移"而非绝对坐标
- **Focal Loss**：给难分样本更高权重，缓解正负 1:1000 的不均衡

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
import torchvision
from PIL import Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import _resolve_pet_root, _collect_pet_samples, CLASS_NAMES as SEG_NAMES
from common.utils import set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torchvision:", torchvision.__version__, "| torch:", torch.__version__)

COCO_NAMES = ["__bg__","person","bicycle","car","motorcycle","airplane","bus","train","truck","boat","traffic light","fire hydrant","__11__","stop sign","parking meter","bench","bird","cat","dog","horse","sheep","cow","elephant","bear","zebra","giraffe","__27__","backpack","umbrella","__30__","__31__","handbag","tie","suitcase","frisbee","skis","snowboard","sports ball","kite","baseball bat","baseball glove","skateboard","surfboard","tennis racket","bottle","__46__","wine glass","cup","fork","knife","spoon","bowl","banana","apple","sandwich","orange","broccoli","carrot","hot dog","pizza","donut","cake","chair","couch","potted plant","bed","__64__","dining table","__66__","__67__","toilet","__69__","tv","laptop","mouse","remote","keyboard","cell phone","microwave","oven","toaster","sink","refrigerator","__83__","book","clock","vase","scissors","teddy bear","hair drier","toothbrush"]


## 2. 数据：4 张 Pet 图（推理样本，不做训练）

复用 01 站已下载的 Oxford-IIIT Pet 原图（`images/*.jpg`），挑 4 张品种各异的猫狗图——COCO 预训练模型能直接认出 cat/dog/person 等，足够演示通用检测。

In [ ]:
pet_root = _resolve_pet_root(str(ROOT / "data"))
names = _collect_pet_samples(pet_root)
# 固定挑 4 张：两猫两狗，品种分散
pick = [n for n in names if any(k in n for k in ["Abyssinian", "Bengal", "beagle", "pomeranian"])]
if len(pick) < 4:
    pick = names[:: max(1, len(names)//4)][:4]
else:
    pick = pick[:4]
print("选中:", pick)

pil_images = []
for n in pick:
    p = pet_root / "images" / f"{n}.jpg"
    pil_images.append(Image.open(p).convert("RGB"))

# fig0：4 张原图
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, im, n in zip(axes, pil_images, pick):
    ax.imshow(im)
    ax.axis("off")
    ax.set_title(n[:18], fontsize=8)
plt.suptitle("推理样本：4 张 Pet 原图（COCO 预训练直接可检）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. 模型定稿：Faster R-CNN / RetinaNet / Mask R-CNN（torchvision 预训练）

三模型同 backbone 族（ResNet-50-FPN），同图同阈值（0.5）对比——预训练权重直接推理，不训练。

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, retinanet_resnet50_fpn, maskrcnn_resnet50_fpn
from torchvision.transforms.functional import to_tensor

def load_detector(kind: str):
    if kind == "faster":
        m = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    elif kind == "retina":
        m = retinanet_resnet50_fpn(weights="DEFAULT")
    elif kind == "mask":
        m = maskrcnn_resnet50_fpn(weights="DEFAULT")
    else:
        raise ValueError(kind)
    m.eval().to(DEVICE)
    return m

faster = load_detector("faster")
retina = load_detector("retina")
maskm = load_detector("mask")
print("权重加载完成：faster / retina / mask 已就绪")

# 参数量（仅 backbone+head，不含后处理，作量级参考）
for name, m in [("Faster R-CNN", faster), ("RetinaNet", retina), ("Mask R-CNN", maskm)]:
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:14s} 参数量 ~ {n_params/1e6:.1f}M")


## 4. 主实验：同图推理对比（Faster vs RetinaNet，阈值 0.5）

In [ ]:
SCORE_TH = 0.5

def infer(model, pil_list, thresh=0.5):
    out = []
    with torch.no_grad():
        for im in pil_list:
            t = to_tensor(im).to(DEVICE)
            pred = model([t])[0]
            keep = pred["scores"] >= thresh
            out.append({k: v[keep].cpu() for k, v in pred.items() if k in ("boxes","labels","scores")})
            if "masks" in pred:
                out[-1]["masks"] = pred["masks"][keep].cpu()
    return out

pred_f = infer(faster, pil_images, SCORE_TH)
pred_r = infer(retina, pil_images, SCORE_TH)

for i, n in enumerate(pick):
    f, r = pred_f[i], pred_r[i]
    flab = [COCO_NAMES[int(l)] if int(l)<len(COCO_NAMES) else str(int(l)) for l in f["labels"]]
    rlab = [COCO_NAMES[int(l)] if int(l)<len(COCO_NAMES) else str(int(l)) for l in r["labels"]]
    print(f"[{n[:20]:20s}] Faster: {len(f['boxes'])}  {list(zip(flab, [round(float(s),2) for s in f['scores']]))}")
    print(f"{'':22s} Retina: {len(r['boxes'])}  {list(zip(rlab, [round(float(s),2) for s in r['scores']]))}")


In [ ]:
def draw_detections(ax, pil_im, pred, title=""):
    ax.imshow(pil_im)
    ax.axis("off")
    ax.set_title(title, fontsize=9)
    cmap = plt.cm.get_cmap("tab10")
    for idx in range(len(pred["boxes"])):
        x1, y1, x2, y2 = pred["boxes"][idx].tolist()
        lab = int(pred["labels"][idx])
        name = COCO_NAMES[lab] if lab < len(COCO_NAMES) else str(lab)
        sc = float(pred["scores"][idx])
        color = cmap(idx % 10)
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(0, y1-4), f"{name} {sc:.2f}", fontsize=7, color="white", bbox=dict(facecolor=color, alpha=0.85, pad=1, edgecolor="none"))

# fig1：2×4 网格，上 Faster 下 Retina
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for c in range(4):
    draw_detections(axes[0, c], pil_images[c], pred_f[c], f"Faster  {pick[c][:14]}")
    draw_detections(axes[1, c], pil_images[c], pred_r[c], f"Retina  {pick[c][:14]}")
plt.suptitle("Faster R-CNN vs RetinaNet（同阈值 0.5，COCO 预训练）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig1_faster_vs_retina.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：柱状——两模型检出数对比
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(4)
w = 0.35
cf = [len(p["boxes"]) for p in pred_f]
cr = [len(p["boxes"]) for p in pred_r]
b1 = ax.bar(x - w/2, cf, w, label="Faster R-CNN", color="#4C72B0")
b2 = ax.bar(x + w/2, cr, w, label="RetinaNet", color="#DD8452")
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05, str(int(b.get_height())), ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([p[:14] for p in pick], fontsize=8)
ax.set_ylabel("检出数 (score≥0.5)")
ax.set_title(f"检出数对比（Faster 总计 {sum(cf)} / Retina 总计 {sum(cr)}）")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig2_counts.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 手写 NMS：IoU + 贪心消重（现场验证）

用 6 个合成框演示：重叠的低分框被 NMS 删掉，与 `torchvision.ops.nms` 对照一致。

In [ ]:
def box_iou(a, b):
    # a,b: [x1,y1,x2,y2]
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area_a = (a[2]-a[0])*(a[3]-a[1]); area_b = (b[2]-b[0])*(b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-9)

def nms_numpy(boxes, scores, iou_th=0.5):
    order = np.argsort(-scores)
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        rest = []
        for j in order[1:]:
            if box_iou(boxes[i], boxes[j]) <= iou_th:
                rest.append(j)
        order = np.array(rest)
    return keep

# 合成 6 框：两簇重叠
boxes = np.array([[10,10,60,60],[15,15,65,65],[14,12,62,58],[100,100,160,160],[105,105,165,165],[200,200,240,240]], dtype=float)
scores = np.array([0.95, 0.90, 0.88, 0.92, 0.85, 0.80])
keep = nms_numpy(boxes, scores, 0.5)
print("手写 NMS keep:", keep, "->", [round(float(scores[i]),2) for i in keep])
# 与 torchvision 对照
import torchvision.ops as ops
keep_tv = ops.nms(torch.as_tensor(boxes), torch.as_tensor(scores), 0.5).tolist()
print("torchvision NMS keep:", keep_tv, "一致:", keep == keep_tv)

# 可视化：左 NMS 前（全框半透明），右 NMS 后
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, idxs, ttl in [(axes[0], list(range(6)), "NMS 前（6 框）"), (axes[1], keep, "NMS 后（IoU>0.5 消重）")]:
    ax.set_xlim(0, 260); ax.set_ylim(260, 0)
    ax.set_aspect("equal"); ax.set_title(ttl)
    cmap = plt.cm.get_cmap("tab10")
    for k in idxs:
        x1,y1,x2,y2 = boxes[k]
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2, edgecolor=cmap(k%10), facecolor=cmap(k%10), alpha=0.22))
        ax.text(x1, y1-3, f"{scores[k]:.2f}", fontsize=8, color=cmap(k%10))
plt.suptitle("手写 NMS：重叠低分框被抑制（与 torchvision 一致）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig3_nms.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. 实例分割：Mask R-CNN（检测+掩码）

同一 4 图，Mask R-CNN 在框之外再给像素掩码——实例分割 = 检测的框 + 分割的形。

In [ ]:
pred_m = infer(maskm, pil_images, SCORE_TH)
for i, n in enumerate(pick):
    pm = pred_m[i]
    mlab = [COCO_NAMES[int(l)] if int(l)<len(COCO_NAMES) else str(int(l)) for l in pm["labels"]]
    print(f"[{n[:20]:20s}] Mask R-CNN: {len(pm['boxes'])}  {list(zip(mlab, [round(float(s),2) for s in pm['scores']]))}")

# fig4：Mask 叠加（框 + 半透明掩码）
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, im, pred, n in zip(axes, pil_images, pred_m, pick):
    ax.imshow(im); ax.axis("off")
    ax.set_title(n[:16], fontsize=8)
    cmap = plt.cm.get_cmap("tab10")
    for idx in range(len(pred["boxes"])):
        x1,y1,x2,y2 = pred["boxes"][idx].tolist()
        lab = int(pred["labels"][idx]); name = COCO_NAMES[lab] if lab < len(COCO_NAMES) else str(lab)
        sc = float(pred["scores"][idx])
        color = cmap(idx % 10)
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2, edgecolor=color, facecolor="none"))
        ax.text(x1, max(0,y1-4), f"{name} {sc:.2f}", fontsize=6, color="white", bbox=dict(facecolor=color, alpha=0.85, pad=1, edgecolor="none"))
        # 掩码半透明叠加
        if "masks" in pred:
            m = pred["masks"][idx, 0].numpy()
            # resize mask to image size if needed (maskm 输出与输入同分辨率附近，PIL 图尺寸与张量一致时直接可用；否则插值)
            if m.shape != (im.size[1], im.size[0]):
                m = np.array(Image.fromarray((m*255).astype(np.uint8)).resize(im.size, Image.NEAREST)) / 255.0
            ax.imshow(np.ma.masked_where(m < 0.5, m), cmap=plt.cm.get_cmap("tab10", 10), alpha=0.32, vmin=0, vmax=10)
plt.suptitle("Mask R-CNN：框 + 实例掩码（阈值 0.5）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig4_maskrcnn.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. 总结与下一步

**本项目收获**

1. two-stage vs one-stage 的取舍：Faster 精、Retina 快且用 Focal 抗不均衡
2. 检测三件套现场验证：Anchor 思路 + 框 IoU + 手写 NMS（与 torchvision 一致）
3. 预训练推理全流程与阈值对检出数的影响
4. 实例分割 = 检测+分割，Mask R-CNN 一次给出框与形

**下一步**：`03_YOLO_Concept`——YOLO 的 grid 思想与 NMS 的更多玩法（不做大训练）。